# Liiga 2026-27 — daily forecast

Runs the model **inside Snowflake**, independently of the Mac: it
fetches liiga.fi itself, rebuilds `LIIGA.RAW`/`LIIGA.MODEL`, and the
Streamlit app reads the result.

The code comes from `LIIGA.CODE.LIIGA_REPO`, so shipping a model
change is `git push` + this notebook's next run.


In [ ]:
# Bootstrap: pull the model out of the git repository.
#
# Only the code and the curated inputs are copied. data/raw is deliberately
# skipped -- it is 2800 cached per-game JSONs, and the season-level liiga.fi
# endpoint returns the same facts in six calls.
import os, shutil, sys
from snowflake.snowpark.context import get_active_session

session = get_active_session()

REPO = "@LIIGA.CODE.LIIGA_REPO/branches/main"
ROOT = "/tmp/liiga"
DATA_FILES = ["crowd_predictions_2026_27.txt", "external_players.csv",
              "external_raw.txt", "external_returnees_raw.txt",
              "goalies_raw.txt", "league_factors.csv",
              "rosters_2026_27.csv", "transfers_2026_27.txt"]
SQL_FILES = ["stg_games.sql", "team_game_log.sql", "team_season.sql",
             "player_season_scoring.sql"]

shutil.rmtree(ROOT, ignore_errors=True)
for d in ("", "/src/liiga", "/src/liiga/sql", "/data", "/data/raw"):
    os.makedirs(ROOT + d, exist_ok=True)

session.sql("ALTER GIT REPOSITORY LIIGA.CODE.LIIGA_REPO FETCH").collect()
session.file.get(REPO + "/config.yaml", ROOT)
session.file.get(REPO + "/src/liiga/", ROOT + "/src/liiga")
for f in SQL_FILES:
    session.file.get(REPO + "/src/liiga/sql/" + f, ROOT + "/src/liiga/sql")
for f in DATA_FILES:
    session.file.get(REPO + "/data/" + f, ROOT + "/data")

sys.path.insert(0, ROOT + "/src")
print("repo:", len(os.listdir(ROOT + "/src/liiga")), "modules")


In [ ]:
# Where to read from and write to. Kept in a table rather than hardcoded so a
# dry run against preseason games can be pointed at throwaway schemas without
# editing (and redeploying) the notebook.
rows = session.sql("SELECT tournament, raw_schema, model_schema "
                   "FROM LIIGA.CODE.PIPELINE_SETTINGS").collect()
TOURNAMENT, RAW_SCHEMA, MODEL_SCHEMA = (
    (rows[0]["TOURNAMENT"], rows[0]["RAW_SCHEMA"], rows[0]["MODEL_SCHEMA"])
    if rows else ("runkosarja", "RAW", "MODEL"))

import yaml
with open(ROOT + "/config.yaml", encoding="utf-8") as fh:
    cfg_file = yaml.safe_load(fh)
cfg_file["database"]["target"] = "snowflake"
cfg_file["ingestion"]["tournament"] = TOURNAMENT
cfg_file["snowflake_sync"]["raw_schema"] = RAW_SCHEMA
cfg_file["snowflake_sync"]["model_schema"] = MODEL_SCHEMA
with open(ROOT + "/config.yaml", "w", encoding="utf-8") as fh:
    yaml.safe_dump(cfg_file, fh)

# The transforms create and read unqualified table names, so both schemas have
# to resolve. A notebook is a session and may do this; a stored procedure may
# not -- which is why this is a notebook.
session.sql("USE SCHEMA LIIGA." + MODEL_SCHEMA).collect()
session.sql("ALTER SESSION SET SEARCH_PATH = 'LIIGA." + MODEL_SCHEMA
            + ", LIIGA." + RAW_SCHEMA + "'").collect()
print(TOURNAMENT, "->", "LIIGA." + RAW_SCHEMA, "/", "LIIGA." + MODEL_SCHEMA)


In [ ]:
# The daily run: fetch liiga.fi, rebuild the tables, re-forecast.
from liiga.config import load_config
from liiga.ingest import ingest_all
from liiga.pipeline import forecast, persist
from liiga.team_strength import build_team_strength
from liiga.transform import run_transforms

counts = ingest_all(force=True)
print("ingest:", counts)

print("transforms:", run_transforms())

build_team_strength()
print("team_strength rebuilt")

res = forecast(cfg=load_config())
persist(res)

top = res["standings"].sort_values("proj_rank").head(5)
print(f"{res['n_played']}/{res['n_total']} games played, "
      f"crowd weight {res['crowd_weight']:.2f}")
for _, r in top.iterrows():
    print(f"  {int(r['proj_rank']):2}. {r['team']:9} {r['mean_points']:.0f}")
